In [6]:
import pandas as pd
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



In [7]:
users_df = pd.read_csv(
    r"C:\Users\gangu\Downloads\users.csv"
)

jobs_df = pd.read_csv(
    r"C:\Users\gangu\Downloads\jobs.csv"
)

interactions_df = pd.read_csv(
    r"C:\Users\gangu\Downloads\interactions.csv"
)

In [8]:
users_df = users_df.fillna("")
jobs_df = jobs_df.fillna("")
interactions_df = interactions_df.fillna("")

In [9]:
jobs_df["job_text"] = (
    jobs_df["title"].astype(str) + " " +
    jobs_df["skills"].astype(str) + " " +
    jobs_df["location"].astype(str) + " " +
    jobs_df["experience_level"].astype(str)
)

In [10]:
vectorizer = TfidfVectorizer(
    stop_words="english"
)

job_vectors = vectorizer.fit_transform(
    jobs_df["job_text"]
)

In [11]:
with open("tfidf_vectorizer.pkl", "wb") as file:
    pickle.dump(vectorizer, file)


with open("job_vectors.pkl", "wb") as file:
    pickle.dump(job_vectors, file)


with open("jobs_data.pkl", "wb") as file:
    pickle.dump(jobs_df, file)


print("Model files saved successfully")

Model files saved successfully


In [12]:
print("\nHybrid Job Recommendation System\n")

skills = input("Enter your skills: ")

role = input("Enter preferred role: ")

location = input("Enter preferred location: ")

experience = input("Enter experience in years: ")


Hybrid Job Recommendation System



Enter your skills:  python
Enter preferred role:  developer
Enter preferred location:  kolkata
Enter experience in years:  7


In [13]:
user_profile = (
    skills + " " +
    role + " " +
    location + " " +
    experience
)


In [15]:
user_vector = vectorizer.transform(
    [user_profile]
)

In [16]:
similarity_scores = cosine_similarity(
    user_vector,
    job_vectors
)

similarity_scores = similarity_scores.flatten()

In [17]:
interaction_matrix = interactions_df.pivot_table(
    index="user_id",
    columns="job_id",
    values="duration_sec",
    aggfunc="mean",
    fill_value=0
)



In [18]:
collaborative_scores = {}

for _, row in interactions_df.iterrows():

    job_id = row["job_id"]

    if job_id not in collaborative_scores:
        collaborative_scores[job_id] = 0

    collaborative_scores[job_id] += row["duration_sec"]


In [19]:
final_scores = []

for index in range(len(jobs_df)):

    job_id = jobs_df.iloc[index]["job_id"]

    content_score = similarity_scores[index]

    collaborative_score = collaborative_scores.get(job_id, 0)

    final_score = content_score + (
        collaborative_score * 0.0001
    )

    final_scores.append(final_score)


jobs_df["recommendation_score"] = final_scores


In [20]:
recommended_jobs = jobs_df.sort_values(
    by="recommendation_score",
    ascending=False
)


top_jobs = recommended_jobs[
    [
        "title",
        "company",
        "location",
        "skills",
        "recommendation_score"
    ]
].head(10)


print("\nTop Recommended Jobs\n")

print(top_jobs.to_string(index=False))


Top Recommended Jobs

               title      company location                  skills  recommendation_score
Full Stack Developer    Accenture  Kolkata           Python,SQL,ML              0.884856
  Frontend Developer    Accenture  Kolkata           Python,SQL,ML              0.849871
    Mobile Developer        Wipro  Kolkata           Python,SQL,ML              0.837925
    Mobile Developer     Flipkart  Kolkata           Python,SQL,ML              0.834092
  Frontend Developer          HCL  Kolkata TensorFlow,MLOps,Python              0.812691
    Mobile Developer    Microsoft  Kolkata           Python,SQL,ML              0.809325
  Frontend Developer TechMahindra  Kolkata           Python,SQL,ML              0.806720
    Mobile Developer      Infosys  Kolkata           Python,SQL,ML              0.787425
    Mobile Developer     Razorpay  Kolkata           Python,SQL,ML              0.787042
Full Stack Developer TechMahindra  Kolkata TensorFlow,MLOps,Python              0.76953